# Case Method Data Mining — Deteksi Transaksi Fraud Bank dengan LightGBM

**Nama / NIM:** ...
**Algoritma:** LightGBM (Light Gradient Boosting Machine)

Urutan notebook ini mengikuti kerangka laporan:

| Bagian notebook | Subbab laporan |
|---|---|
| Bab 2 | Dataset (2.1–2.3) |
| Bab 3 | Metode (3.2–3.5) |
| Bab 4 | Hasil dan Pembahasan (4.1–4.7) |
| Bab 5 | Ringkasan angka untuk Kesimpulan dan Rekomendasi |

Semua grafik (`.png`) dan tabel (`.csv`) otomatis disimpan di folder **`hasil/`** dengan nama file sesuai nomor subbab, misalnya `4.2_perbandingan_model.csv`.

**Cara menjalankan:** pilih menu *Run All* (Colab: `Runtime → Run all`, VS Code/Jupyter: `Run All`). Seluruh proses memakan waktu sekitar 2–5 menit.

## 0. Persiapan
Jika memakai **Google Colab**, hapus tanda `#` di sel berikut lalu jalankan. Jika menjalankan di laptop dan library sudah dipasang lewat terminal, sel ini boleh dilewati.

In [ ]:
# %pip install -q lightgbm shap kagglehub

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore", message=".*eval_set.*")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

OUT = "hasil"
os.makedirs(OUT, exist_ok=True)

def simpan_gambar(nama):
    """Simpan grafik yang sedang aktif ke folder hasil/."""
    plt.savefig(f"{OUT}/{nama}.png", dpi=150, bbox_inches="tight")

def simpan_tabel(df_, nama):
    """Simpan tabel ke folder hasil/ dalam format CSV."""
    df_.to_csv(f"{OUT}/{nama}.csv")

print("Library siap. LightGBM versi", lgb.__version__)

### Memuat dataset
Program mencari file **`creditcard.csv`** di folder yang sama dengan notebook. Jika tidak ada, program mencoba mengunduhnya otomatis dari Kaggle lewat `kagglehub`.

Sumber dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

In [ ]:
def muat_dataset():
    if os.path.exists("creditcard.csv"):
        print("Memakai creditcard.csv dari folder notebook.")
        return pd.read_csv("creditcard.csv")
    try:
        import kagglehub
        print("creditcard.csv tidak ditemukan, mengunduh dari Kaggle...")
        folder = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        return pd.read_csv(os.path.join(folder, "creditcard.csv"))
    except Exception as e:
        raise FileNotFoundError(
            "Dataset tidak ditemukan. Unduh creditcard.csv dari "
            "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud lalu letakkan "
            f"satu folder dengan notebook ini. (Detail: {e})")

df_raw = muat_dataset()
print("Ukuran data:", df_raw.shape)
df_raw.head()

---
# BAB 2 — Dataset
## 2.2 Deskripsi Fitur

In [ ]:
deskripsi = pd.DataFrame({
    "Fitur": ["Time", "V1 – V28", "Amount", "Class"],
    "Tipe": ["Numerik", "Numerik", "Numerik", "Kategorikal (0/1)"],
    "Keterangan": [
        "Detik sejak transaksi pertama dalam dataset",
        "Hasil transformasi PCA; makna aslinya dirahasiakan demi privasi nasabah",
        "Nominal transaksi",
        "Label: 1 = fraud, 0 = normal",
    ],
}).set_index("Fitur")
simpan_tabel(deskripsi, "2.2_deskripsi_fitur")
deskripsi

## 2.3 Karakteristik Data

In [ ]:
karakteristik = pd.Series({
    "Jumlah transaksi": len(df_raw),
    "Jumlah fitur (tanpa label)": df_raw.shape[1] - 1,
    "Total missing value": int(df_raw.isna().sum().sum()),
    "Jumlah baris duplikat": int(df_raw.duplicated().sum()),
    "Jumlah transaksi normal": int((df_raw["Class"] == 0).sum()),
    "Jumlah transaksi fraud": int((df_raw["Class"] == 1).sum()),
    "Persentase fraud (%)": round(df_raw["Class"].mean() * 100, 4),
}, name="Nilai").to_frame()
simpan_tabel(karakteristik, "2.3_karakteristik_data")
karakteristik

In [ ]:
dist = df_raw["Class"].value_counts().rename(index={0: "Normal", 1: "Fraud"})
plt.figure(figsize=(5, 4))
ax = dist.plot(kind="bar", color=["#4C72B0", "#C44E52"], rot=0)
ax.set_yscale("log")
for i, v in enumerate(dist):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_title("Distribusi Kelas Transaksi (skala log)")
ax.set_ylabel("Jumlah transaksi")
simpan_gambar("2.3_distribusi_kelas")
plt.show()

---
# BAB 3 — Metode
## 3.2 Preprocessing
1. Menghapus baris duplikat.
2. Mengubah `Time` menjadi **jam transaksi** (`Hour`, 0–23), lalu membuang `Time`. Alasannya, `Time` hanya berisi detik sejak transaksi pertama dalam 2 hari, sehingga tidak bermakna untuk data baru.
3. Membagi data secara **stratified**: train 60%, validation 20%, test 20%.
   - *Train*: melatih model
   - *Validation*: early stopping, memilih model terbaik, dan memilih threshold
   - *Test*: penilaian akhir (hanya dipakai sekali)
4. **Tidak ada normalisasi untuk LightGBM** karena berbasis pohon. Normalisasi hanya dipakai pada Logistic Regression.

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
df["Hour"] = (df["Time"] // 3600 % 24).astype(int)
df = df.drop(columns="Time")

X = df.drop(columns="Class")
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)

pembagian = pd.DataFrame({
    "Jumlah transaksi": [len(y_train), len(y_val), len(y_test)],
    "Jumlah fraud": [y_train.sum(), y_val.sum(), y_test.sum()],
    "Persentase fraud (%)": [y_train.mean()*100, y_val.mean()*100, y_test.mean()*100],
}, index=["Train (60%)", "Validation (20%)", "Test (20%)"])
print(f"Duplikat dihapus: {len(df_raw) - len(df):,} baris")
simpan_tabel(pembagian, "3.2_pembagian_data")
pembagian

## 3.3 – 3.5 Pelatihan Model

**3.3 LightGBM** membangun banyak decision tree kecil secara berurutan; setiap pohon baru memperbaiki kesalahan pohon sebelumnya (*gradient boosting*). Pohon tumbuh secara *leaf-wise* (dari daun yang paling menurunkan error) dan nilai fitur dikelompokkan ke dalam *histogram bin*, sehingga cepat untuk data besar.

**3.4 Penanganan data tidak seimbang:** parameter `scale_pos_weight` memberi bobot lebih besar pada kelas fraud, lalu threshold keputusan dipilih ulang pada data validation.

**3.5 Model pembanding:**
- *Dummy* — selalu menebak "normal", untuk menunjukkan bahwa akurasi menyesatkan
- *Logistic Regression* — model linear sederhana sebagai pembanding

In [ ]:
# --- Model pembanding ---
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

logreg = make_pipeline(StandardScaler(),
                       LogisticRegression(class_weight="balanced", max_iter=2000))
logreg.fit(X_train, y_train)
print("Dummy dan Logistic Regression selesai dilatih.")

In [ ]:
# --- LightGBM ---
neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
spw = float(np.sqrt(neg / pos))   # akar rasio: bobot besar tanpa membuat false positive melonjak

base_params = dict(
    n_estimators=2000,        # batas atas; jumlah pohon sebenarnya ditentukan early stopping
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    verbose=-1,
)

def latih_lgbm(**tambahan):
    model = lgb.LGBMClassifier(**base_params, **tambahan)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              eval_metric="average_precision",
              callbacks=[lgb.early_stopping(100, verbose=False)])
    return model

lgbm_plain = latih_lgbm()
lgbm_weighted = latih_lgbm(scale_pos_weight=spw)

hyper = pd.DataFrame({
    "Nilai": [base_params["learning_rate"], base_params["num_leaves"], base_params["min_child_samples"],
              base_params["subsample"], base_params["colsample_bytree"], base_params["reg_lambda"],
              "100 ronde (metrik: average precision)", f"{spw:.2f}",
              lgbm_plain.best_iteration_, lgbm_weighted.best_iteration_],
    "Keterangan": ["Besar langkah tiap pohon baru", "Maksimum daun per pohon",
                   "Minimum data per daun (mencegah overfit)", "Porsi data acak per pohon",
                   "Porsi fitur acak per pohon", "Regularisasi L2",
                   "Berhenti jika validation tidak membaik", f"Akar rasio normal:fraud ({neg/pos:.0f}:1)",
                   "Jumlah pohon terbaik — LightGBM tanpa bobot",
                   "Jumlah pohon terbaik — LightGBM + scale_pos_weight"],
}, index=["learning_rate", "num_leaves", "min_child_samples", "subsample", "colsample_bytree",
          "reg_lambda", "early_stopping", "scale_pos_weight", "best_iteration (tanpa bobot)",
          "best_iteration (dengan bobot)"])
simpan_tabel(hyper, "3.4_hyperparameter")
hyper

---
# BAB 4 — Hasil dan Pembahasan
## 4.1 Pola Transaksi Fraud

In [ ]:
nominal = df.groupby("Class")["Amount"].describe().rename(index={0: "Normal", 1: "Fraud"})
simpan_tabel(nominal, "4.1_statistik_nominal")
display(nominal)

plt.figure(figsize=(6, 4))
sns.boxplot(data=df.assign(Kelas=df["Class"].map({0: "Normal", 1: "Fraud"})),
            x="Kelas", y="Amount", hue="Kelas", legend=False, showfliers=False,
            palette={"Normal": "#4C72B0", "Fraud": "#C44E52"})
plt.title("Nominal Transaksi: Normal vs Fraud (tanpa outlier)")
simpan_gambar("4.1_nominal_transaksi")
plt.show()

In [ ]:
fraud_per_jam = df.groupby("Hour")["Class"].mean() * 100
plt.figure(figsize=(8, 4))
fraud_per_jam.plot(marker="o", color="#C44E52")
plt.title("Persentase Fraud per Jam Transaksi")
plt.xlabel("Jam"); plt.ylabel("% transaksi fraud")
plt.xticks(range(24))
simpan_gambar("4.1_fraud_per_jam")
plt.show()

top_jam = fraud_per_jam.sort_values(ascending=False).head(3)
print("3 jam dengan persentase fraud tertinggi:")
for jam, p in top_jam.items():
    print(f"  jam {jam:02d}:00 – {jam:02d}:59 -> {p:.3f}% fraud")

In [ ]:
korelasi = df.corr()["Class"].drop("Class").sort_values(key=np.abs, ascending=False)
plt.figure(figsize=(6, 4))
korelasi.head(10).plot(kind="barh", color=["#C44E52" if v < 0 else "#4C72B0" for v in korelasi.head(10)])
plt.gca().invert_yaxis()
plt.title("10 Fitur dengan Korelasi Terkuat terhadap Fraud")
plt.xlabel("Koefisien korelasi")
simpan_gambar("4.1_korelasi_fitur")
plt.show()

## 4.2 Perbandingan Performa Model
Metrik dihitung pada **data test** dengan threshold default 0,5. Karena fraud sangat sedikit, **PR-AUC** dipakai sebagai metrik utama, bukan akurasi.

In [ ]:
def evaluasi(model, X_, y_, threshold=0.5):
    proba = model.predict_proba(X_)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_, pred),
        "Precision": precision_score(y_, pred, zero_division=0),
        "Recall": recall_score(y_, pred),
        "F1": f1_score(y_, pred),
        "ROC-AUC": roc_auc_score(y_, proba),
        "PR-AUC": average_precision_score(y_, proba),
    }

models = {
    "Dummy (selalu normal)": dummy,
    "Logistic Regression": logreg,
    "LightGBM tanpa bobot": lgbm_plain,
    "LightGBM + scale_pos_weight": lgbm_weighted,
}
perbandingan = pd.DataFrame({n: evaluasi(m, X_test, y_test) for n, m in models.items()}).T
simpan_tabel(perbandingan, "4.2_perbandingan_model")
perbandingan

## 4.3 Pengaruh Threshold
Model LightGBM terbaik dipilih berdasarkan **PR-AUC di data validation** (bukan test, supaya penilaian akhir tetap jujur). Threshold kemudian dipilih pada data validation dengan memaksimalkan F1, lalu diterapkan ke data test.

In [ ]:
pr_val = {
    "LightGBM tanpa bobot": average_precision_score(y_val, lgbm_plain.predict_proba(X_val)[:, 1]),
    "LightGBM + scale_pos_weight": average_precision_score(y_val, lgbm_weighted.predict_proba(X_val)[:, 1]),
}
nama_terbaik = max(pr_val, key=pr_val.get)
best_model = models[nama_terbaik]
print("PR-AUC validation:", {k: round(v, 4) for k, v in pr_val.items()})
print("Model terbaik:", nama_terbaik)

proba_val = best_model.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, proba_val)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = float(thr[np.argmax(f1s)])
print(f"Threshold terbaik (dipilih di validation): {best_thr:.3f}")

pengaruh_thr = pd.DataFrame({
    "Threshold 0,50": evaluasi(best_model, X_test, y_test, 0.5),
    f"Threshold {best_thr:.2f}".replace(".", ","): evaluasi(best_model, X_test, y_test, best_thr),
}).T
simpan_tabel(pengaruh_thr, "4.3_pengaruh_threshold")
pengaruh_thr

## 4.4 Confusion Matrix dan Precision–Recall Curve

In [ ]:
proba_test = best_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= best_thr).astype(int)
cm = confusion_matrix(y_test, pred_test)
tn, fp_n, fn_n, tp_n = cm.ravel()

plt.figure(figsize=(5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Fraud"]).plot(cmap="Blues", colorbar=False, ax=plt.gca())
plt.gca().grid(False)
plt.title(f"Confusion Matrix — {nama_terbaik}\n(data test, threshold {best_thr:.2f})")
simpan_gambar("4.4_confusion_matrix")
plt.show()

print(f"True Negative  (normal, diprediksi normal): {tn:,}")
print(f"False Positive (normal, diprediksi fraud) : {fp_n:,}  -> nasabah normal ikut terganggu")
print(f"False Negative (fraud, diprediksi normal) : {fn_n:,}  -> fraud yang lolos")
print(f"True Positive  (fraud, diprediksi fraud)  : {tp_n:,}  -> fraud yang tertangkap")

In [ ]:
plt.figure(figsize=(6.5, 4.5))
for n, m in list(models.items())[1:]:
    p_ = m.predict_proba(X_test)[:, 1]
    p, r, _ = precision_recall_curve(y_test, p_)
    plt.plot(r, p, label=f"{n} (PR-AUC={average_precision_score(y_test, p_):.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision–Recall Curve (data test)")
plt.legend(loc="lower left")
simpan_gambar("4.4_pr_curve")
plt.show()

## 4.5 Faktor Paling Berpengaruh
1. **Feature importance (gain)** — seberapa besar kontribusi tiap fitur dalam menurunkan error.
2. **SHAP** — menunjukkan *arah* pengaruh. Titik merah = nilai fitur tinggi, biru = rendah. Titik di kanan garis tengah mendorong prediksi ke arah fraud.

In [ ]:
imp = pd.Series(best_model.booster_.feature_importance(importance_type="gain"),
                index=X_train.columns, name="Gain").sort_values(ascending=False)
simpan_tabel(imp.to_frame(), "4.5_feature_importance")

plt.figure(figsize=(7, 5))
imp.head(15).plot(kind="barh", color="#4C72B0")
plt.gca().invert_yaxis()
plt.title(f"15 Fitur Terpenting — {nama_terbaik} (gain)")
plt.xlabel("Total gain")
simpan_gambar("4.5_feature_importance")
plt.show()

In [ ]:
# Sampel: semua fraud di data test + maksimal 2.000 transaksi normal, agar SHAP cepat
idx_fraud = y_test[y_test == 1].index
idx_normal = y_test[y_test == 0].sample(min(2000, int((y_test == 0).sum())),
                                       random_state=RANDOM_STATE).index
X_shap = X_test.loc[idx_fraud.union(idx_normal)]

explainer = shap.TreeExplainer(best_model)
sv = explainer(X_shap)
if sv.values.ndim == 3:   # beberapa versi SHAP mengembalikan nilai per kelas
    sv = sv[:, :, 1]

shap.plots.beeswarm(sv, max_display=15, show=False)
plt.title("SHAP — Arah Pengaruh Fitur terhadap Prediksi Fraud")
simpan_gambar("4.5_shap_summary")
plt.show()

## 4.6 Dampak Bisnis

In [ ]:
tp_mask = (pred_test == 1) & (y_test.values == 1)
fn_mask = (pred_test == 0) & (y_test.values == 1)
total_fraud = X_test.loc[y_test == 1, "Amount"].sum()
tertangkap = X_test.loc[tp_mask, "Amount"].sum()
lolos = X_test.loc[fn_mask, "Amount"].sum()
n_normal = int((y_test == 0).sum())

dampak = pd.Series({
    "Total nominal fraud (data test)": total_fraud,
    "Nominal fraud tertangkap": tertangkap,
    "Nominal fraud lolos": lolos,
    "Persentase nominal tertangkap (%)": tertangkap / total_fraud * 100,
    "Jumlah transaksi fraud tertangkap": tp_n,
    "Jumlah transaksi fraud lolos": fn_n,
    "Transaksi normal ikut ditandai fraud": fp_n,
    "Persentase transaksi normal terganggu (%)": fp_n / n_normal * 100,
}, name="Nilai").to_frame()
simpan_tabel(dampak, "4.6_dampak_bisnis")
dampak

## 4.7 Keterbatasan
- Data hanya mencakup **2 hari transaksi pada September 2013** dari nasabah Eropa, sehingga belum tentu mewakili pola fraud saat ini atau di Indonesia.
- Fitur `V1`–`V28` dianonimkan dengan PCA, sehingga faktor penyebab fraud **tidak bisa dijelaskan secara bisnis**; yang dapat ditafsirkan langsung hanya `Amount` dan `Hour`.
- Model diuji pada data dari periode yang sama dengan data latih; pada penerapan nyata, pola fraud berubah dari waktu ke waktu sehingga model perlu dilatih ulang berkala.

---
# BAB 5 — Ringkasan Angka untuk Kesimpulan dan Rekomendasi
Sel berikut merangkum angka-angka penting dari hasil di atas. **Gunakan sebagai bahan**, lalu tulis kesimpulan dan rekomendasi dengan kalimatmu sendiri di laporan.

In [ ]:
hasil_akhir = evaluasi(best_model, X_test, y_test, best_thr)
lr_hasil = perbandingan.loc["Logistic Regression"]
top3 = ", ".join(imp.head(3).index)

print("=== Pertanyaan 1: Seberapa baik LightGBM mendeteksi fraud? ===")
print(f"Model terbaik: {nama_terbaik}, threshold {best_thr:.2f}")
print(f"Recall {hasil_akhir['Recall']*100:.1f}% (fraud tertangkap), "
      f"precision {hasil_akhir['Precision']*100:.1f}%, F1 {hasil_akhir['F1']:.3f}, "
      f"PR-AUC {hasil_akhir['PR-AUC']:.3f}")
print(f"Pembanding Logistic Regression: PR-AUC {lr_hasil['PR-AUC']:.3f}, F1 {lr_hasil['F1']:.3f}")
print(f"Model Dummy: akurasi {perbandingan.loc['Dummy (selalu normal)', 'Accuracy']*100:.2f}% "
      f"tetapi recall 0% -> akurasi tidak layak dipakai")

print("\n=== Pertanyaan 2: Faktor apa yang paling berpengaruh? ===")
print(f"3 fitur terpenting (gain): {top3}")
print(f"Jam dengan persentase fraud tertinggi: {', '.join(f'{j:02d}:00' for j in top_jam.index)}")

print("\n=== Pertanyaan 3: Bahan rekomendasi ===")
print(f"{dampak.loc['Persentase nominal tertangkap (%)', 'Nilai']:.1f}% nominal fraud tertangkap, "
      f"{int(fn_n)} transaksi fraud lolos")
print(f"{int(fp_n)} transaksi normal ikut ditandai fraud "
      f"({dampak.loc['Persentase transaksi normal terganggu (%)', 'Nilai']:.3f}% dari transaksi normal)")

### Contoh arah rekomendasi *(sesuaikan dengan angka di atas)*
1. Pakai model sebagai **skor risiko real-time** untuk setiap transaksi.
2. Terapkan **tindakan bertingkat**: skor sangat tinggi → blokir sementara + verifikasi; skor sedang → verifikasi tambahan; skor rendah → lolos.
3. Sesuaikan threshold dengan **kapasitas tim investigasi** dan toleransi bank terhadap nasabah yang terganggu.
4. **Latih ulang model secara berkala** karena pola fraud terus berubah.
5. **Kombinasikan dengan aturan (rule-based)** yang sudah dimiliki bank.

---
### (Opsional) Unduh folder `hasil/` di Google Colab

In [ ]:
import shutil
shutil.make_archive("hasil_case_method", "zip", OUT)
try:
    from google.colab import files
    files.download("hasil_case_method.zip")
except ImportError:
    print("Bukan di Colab: file hasil_case_method.zip sudah dibuat di folder notebook.")